# Hi-EF Phase 1 canonical multi-seed validation

This is the canonical validation-only run: five fixed seeds times two models on the frozen source-folder-held-out split. It never evaluates the test partition. Do not edit the seed list or training configuration between runs.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/phase1_multiseed_validation')

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)

assert (FEATURES / '01_00059.pt').exists(), 'Attach ptrnghieu/hi-ef-features-v2'
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_validation_matrix.py'),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--features-dir', str(FEATURES),
    '--output-dir', str(OUTPUT),
    '--seeds', '42', '123', '456', '789', '1024',
    '--epochs', '50', '--batch-size', '32', '--workers', '2',
    '--learning-rate', '1e-4', '--weight-decay', '1e-5',
    '--patience', '8', '--d-model', '512',
    '--temporal-layers', '2', '--inter-layers', '2',
    '--dropout', '0.1', '--face-pooling', 'masked'
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

summary = json.loads((OUTPUT / 'validation_matrix_summary.json').read_text())
assert summary['test_evaluated'] is False
assert summary['seeds'] == [42, 123, 456, 789, 1024]
display(pd.read_csv(OUTPUT / 'validation_matrix.csv'))
print(json.dumps(summary['aggregate'], indent=2))

Run only via **Save Version → Save & Run All**. The saved output must contain `validation_matrix_summary.json`, `validation_matrix.csv`, and ten run directories with checkpoints, histories, configs, and validation predictions.